In [3]:
# Run this in a terminal or a Jupyter cell prefixed with ! for pip installs
!pip install pandas numpy scikit-learn sentence-transformers faiss-cpu openai transformers torch networkx spacy matplotlib ipywidgets tiktoken
# optionally for rdflib if you want RDF KG:
!pip install rdflib
# install spacy model
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 1.7 MB/s eta 0:00:08
     --- ------------------------------------ 1.0/12.8 MB 1.8 MB/s eta 0:00:07
     --- ------------------------------------ 1.0/12.8 MB 1.8 MB/s eta 0:00:07
     ---- ----------------------------------- 1.6/12.8 MB 1.4 MB/s eta 0:00:08
     ------ --------------------------------- 2.1/12.8 MB 1.6 MB/s eta 0:00:07
     --------- ------------------------------ 2.9/12.8 MB 1.9 MB/s eta 0:00:06
     ---------- ----------------------------- 3.4/12.8 MB 2.0 MB/s eta 0:00:05
     ----------- ---------------------------- 3.7/12.8 MB 2.0 MB/s eta 0:00:05
     ------------- -------------------------- 4.5/12.8 MB 2.1 MB/s eta 0:00:05
     -------------- ------------------------- 4.7/12.8 MB 2.1 MB/s eta 0:

In [4]:
# Cell 1: basic imports and config
import os
import random
import json
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)


# ML & metrics
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, confusion_matrix)

# Embeddings & retrieval
from sentence_transformers import SentenceTransformer
import faiss

# Graph handling
import networkx as nx
import spacy

# LLM / API
# openai for ChatCompletion / embeddings if you use OpenAI
import openai

# seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Configure OpenAI key if you plan to use it (set as env var)
# export OPENAI_API_KEY='sk-...'
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY:
    openai.api_key = OPENAI_API_KEY

# Load spaCy NER model
nlp = spacy.load("en_core_web_sm")


<h2>Load & inspect dataset(s) </h2>

In [5]:
# Cell 2: load dataset example (adapt path and column names)
data_path = "gai_personalized_learning_dataset.csv"   # change this
df = pd.read_csv(data_path)

# Expected columns examples
print(df.shape)
df.head()


(200, 10)


,user_id,course_id,title,description,difficulty_level,interaction_type,engagement_score,performance_score,timestamp,label
0,U23,C11,Natural Language Processing,Text processing and language models,Intermediate,view,0.70,67,2025-11-12 11:29:50,0
1,U34,C16,Data Science with Python,Advanced techniques and models,Intermediate,view,0.88,41,2025-10-04 11:29:50,1
2,U46,C13,Deep Learning Fundamentals,Foundational concepts and theory,Advanced,click,0.51,93,2026-01-07 11:29:50,1
3,U44,C15,Deep Learning Fundamentals,Text processing and language models,Beginner,quiz_attempt,0.94,91,2025-09-09 11:29:50,1
4,U8,C14,Deep Learning Fundamentals,Hands-on practical course,Beginner,view,0.69,98,2025-11-09 11:29:50,0


<h2> Preprocessing  document creation (for RAG) </h2>

In [6]:
# Cell 3: build documents table for embedding
# Example: combine title + description + content + tags
def make_doc_row(row):
    parts = []
    for col in ["title", "interaction_type", "performance_score"]:
        if col in row and pd.notna(row[col]):
            parts.append(str(row[col]))
    return parts

items = df.drop_duplicates(subset=["course_id"]).copy()
items["doc_text"] = items.apply(make_doc_row, axis=1)
items = items.reset_index(drop=True)

print("Num docs:", len(items))
items.head()


Num docs: 20


,user_id,course_id,title,description,difficulty_level,interaction_type,engagement_score,performance_score,timestamp,label,doc_text
0,U23,C11,Natural Language Processing,Text processing and language models,Intermediate,view,0.70,67,2025-11-12 11:29:50,0,"[Natural Language Processing, view, 67]"
1,U34,C16,Data Science with Python,Advanced techniques and models,Intermediate,view,0.88,41,2025-10-04 11:29:50,1,"[Data Science with Python, view, 41]"
2,U46,C13,Deep Learning Fundamentals,Foundational concepts and theory,Advanced,click,0.51,93,2026-01-07 11:29:50,1,"[Deep Learning Fundamentals, click, 93]"
3,U44,C15,Deep Learning Fundamentals,Text processing and language models,Beginner,quiz_attempt,0.94,91,2025-09-09 11:29:50,1,"[Deep Learning Fundamentals, quiz_attempt, 91]"
4,U8,C14,Deep Learning Fundamentals,Hands-on practical course,Beginner,view,0.69,98,2025-11-09 11:29:50,0,"[Deep Learning Fundamentals, view, 98]"


<h2> Create dense embeddings + FAISS index (retrieval) </h2>

In [7]:
# Create items dataframe for retrieval
items = df.copy()

items["doc_text"] = (
    items["title"] + ". " +
    items["description"] + ". " +
    "Difficulty: " + items["difficulty_level"] + ". " +
    "Course ID: " + items["course_id"]
)


items = items[["course_id", "title", "doc_text"]].drop_duplicates()
items.head()

,course_id,title,doc_text
0,C11,Natural Language Processing,Natural Language Processing. Text processing and language models. Difficulty: Intermediate. Course ID: C11
1,C16,Data Science with Python,Data Science with Python. Advanced techniques and models. Difficulty: Intermediate. Course ID: C16
2,C13,Deep Learning Fundamentals,Deep Learning Fundamentals. Foundational concepts and theory. Difficulty: Advanced. Course ID: C13
3,C15,Deep Learning Fundamentals,Deep Learning Fundamentals. Text processing and language models. Difficulty: Beginner. Course ID: C15
4,C14,Deep Learning Fundamentals,Deep Learning Fundamentals. Hands-on practical course. Difficulty: Beginner. Course ID: C14


In [8]:
embed_model_name = "all-mpnet-base-v2"
embed_model = SentenceTransformer(embed_model_name)

docs = items["doc_text"].fillna("").tolist()

# Compute embeddings
doc_embeddings = embed_model.encode(
    docs,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Normalize for cosine similarity
faiss.normalize_L2(doc_embeddings)

d = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(doc_embeddings)

print("FAISS index size:", index.ntotal)

# Save artifacts
Path("artifacts").mkdir(exist_ok=True)

np.save("artifacts/doc_embeddings.npy", doc_embeddings)
items.to_pickle("artifacts/items.pkl")

faiss.write_index(index, "artifacts/faiss.index")


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

FAISS index size: 192


<h2> Build a simple Knowledge Graph (KG) from metadata / NER </h2>

In [9]:
# Cell 5: build small KG (NetworkX)
G = nx.Graph()
# Add item nodes
for idx, row in items.iterrows():
    item_node = f"item_{row['course_id']}"
    G.add_node(item_node, type="item", title=row.get("title",""))

# Extract entities using spaCy NER from doc_text and add as concept nodes
for idx, row in items.iterrows():
    text = row["doc_text"][:2000]  # limit for speed
    doc = nlp(text)
    ents = set([ent.text.lower() for ent in doc.ents if len(ent.text) > 2])
    item_node = f"item_{row['course_id']}"
    for e in ents:
        concept_node = f"concept_{e}"
        if not G.has_node(concept_node):
            G.add_node(concept_node, type="concept", label=e)
        # add edge item <-> concept
        G.add_edge(item_node, concept_node, relation="mentions")

print("KG nodes:", len(G.nodes()), "edges:", len(G.edges()))
# Save KG
nx.write_graphml(G, "artifacts/knowledge_graph.graphml")


KG nodes: 25 edges: 33


<h2>Graph-RAG retrieval function</h2>

In [10]:
def graph_rag_retrieve(query, top_k=5, kg_expand_k=5):
    # --- FAISS retrieval ---
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    D, I = index.search(q_emb, top_k)

    faiss_docs = [items.iloc[i] for i in I[0] if i != -1]
    faiss_item_nodes = {f"item_{r['course_id']}" for r in faiss_docs}

    # --- KG expansion ---
    doc = nlp(query)
    ents = {ent.text.lower() for ent in doc.ents if len(ent.text) > 2}

    kg_docs = []
    for e in ents:
        concept_node = f"concept_{e}"
        if concept_node in G:
            neighbors = [
                n for n in G.neighbors(concept_node)
                if G.nodes[n]["type"] == "item"
            ]
            for n in neighbors[:kg_expand_k]:
                cid = n.split("_", 1)[1]   # keep as string
                row = items[items["course_id"] == cid]
                if not row.empty:
                    kg_docs.append(row.iloc[0])

    # --- Merge & deduplicate ---
    merged = []
    seen = set()

    for r in faiss_docs + kg_docs:
        key = r["course_id"]
        if key not in seen:
            merged.append(r)
            seen.add(key)

    return merged


In [11]:
results = graph_rag_retrieve("linear regression course basics", top_k=5)

for r in results[:6]:
    print(r["course_id"], "→", r["title"])


C16 → Machine Learning Basics
C1 → Machine Learning Basics
C11 → Machine Learning Basics
C19 → Machine Learning Basics
C17 → Machine Learning Basics


In [12]:
items["title"].value_counts()

title
Data Science with Python       43
Introduction to AI             41
Deep Learning Fundamentals     40
Machine Learning Basics        36
Natural Language Processing    32
Name: count, dtype: int64

<h2>RAG prompt + LLM call (generate answer grounded on retrieved docs)</h2>

In [13]:
# Cell 7: RAG prompt + LLM (OpenAI example)
def build_rag_prompt(query, retrieved_docs, max_chars_per_doc=800):
    ctx_parts = []
    for i, doc in enumerate(retrieved_docs):
        text = str(doc["doc_text"])[:max_chars_per_doc]
        ctx_parts.append(f"[SRC_{i+1}] Title: {doc.get('title','')}\n{text}\n")
    context = "\n\n".join(ctx_parts)
    prompt = f"""You are an educational assistant. Use ONLY the information from the CONTEXT below to answer the USER question.
    
CONTEXT:
{context}

USER QUESTION:
{query}

INSTRUCTIONS:
- Give a concise, pedagogical answer.
- Cite the source(s) inline like [SRC_1].
- If uncertain, say you cannot answer precisely and suggest resources.
- Keep answer to ~250-400 words.
"""
    return prompt

def call_openai_chat(prompt, model="gpt-4o-mini", temperature=0.2, max_tokens=512):
    if OPENAI_API_KEY is None:
        raise RuntimeError("OpenAI API key not configured in OPENAI_API_KEY env var.")
    resp = openai.ChatCompletion.create(
        model=model,
        messages=[
            {"role":"system","content":"You are an expert pedagogical assistant."},
            {"role":"user","content":prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    text = resp["choices"][0]["message"]["content"]
    return text

# Example end-to-end
query = "Explain the difference between supervised and unsupervised learning"
retrieved = graph_rag_retrieve(query, top_k=5)
prompt = build_rag_prompt(query, retrieved)
# ans = call_openai_chat(prompt)   # uncomment to run if key configured
# print(ans)


<h2>Recommendation baselines</h2>

# Content-based recommender (TF-IDF similarity)

In [14]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def content_based_recommend(user_history_item_ids, top_k=10):
    # Get indices of items in user history
    indices = [
        items.index[items["course_id"] == iid][0]
        for iid in user_history_item_ids
    ]

    # Build user profile (mean TF-IDF vector)
    user_vec = tfidf_matrix[indices].mean(axis=0)

    # FIX: convert np.matrix → np.ndarray and ensure 2D
    user_vec = np.asarray(user_vec)

    # Compute cosine similarity
    sims = cosine_similarity(user_vec, tfidf_matrix).flatten()

    # Rank items
    top_idx = sims.argsort()[::-1][:top_k]
    rec_items = items.iloc[top_idx]

    return rec_items


In [15]:
content_based_recommend(
    [items["course_id"].iloc[0]],
    top_k=5
)[["course_id", "title"]]


NameError: name 'tfidf_matrix' is not defined

# Classical ML baseline (RandomForest for next-item / classification)

In [ ]:
# Cell 9: create features for user-item classifier (toy example)
# Build simple aggregated user features
user_agg = df.groupby("user_id").agg(
    total_interactions=("label", "sum"),
    last_ts=("timestamp", "max")
).reset_index()

#Item features (TF-IDF → SVD)
from sklearn.decomposition import TruncatedSVD

SEED = 42

svd = TruncatedSVD(n_components=32, random_state=SEED)
item_svd = svd.fit_transform(tfidf_matrix)  # shape: n_items x 32

items_feat = items.reset_index(drop=True).copy()
items_feat["it_index"] = items_feat.index

#Construction des paires (user, course)
pairs_df = df[["user_id", "course_id", "label"]].copy()

#Jointure des features
#User features
pairs_df = pairs_df.merge(user_agg, on="user_id", how="left")
#Item index
pairs_df = pairs_df.merge(
    items_feat[["course_id", "it_index"]],
    on="course_id",
    how="left"
)
pairs_df = pairs_df.dropna(subset=["it_index"])
pairs_df["it_index"] = pairs_df["it_index"].astype(int)

#Matrice finale X / y
X_user = pairs_df[["total_interactions"]].fillna(0).values
X_item = item_svd[pairs_df["it_index"].values]

X = np.hstack([X_user, X_item])
y = pairs_df["label"].values

#Train / Test + RandomForest
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=SEED,
    n_jobs=-1
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1       :", f1_score(y_test, y_pred, zero_division=0))


# Simple deep baseline (MLP)

In [ ]:
# Cell 10: simple PyTorch MLP (toy)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class SimpleMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256,64),
            nn.ReLU(),
            nn.Linear(64,1),
            nn.Sigmoid()
        )
    def forward(self,x):
        return self.net(x)

in_dim = X_train.shape[1]
model = SimpleMLP(in_dim).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()

# prepare loaders
train_ds = TensorDataset(torch.tensor(X_train,dtype=torch.float32), torch.tensor(y_train,dtype=torch.float32))
test_ds  = TensorDataset(torch.tensor(X_test,dtype=torch.float32), torch.tensor(y_test,dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

# training loop (few epochs)
for epoch in range(8):
    model.train()
    for xb,yb in train_loader:
        xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
        preds = model(xb)
        loss = loss_fn(preds, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    # eval quick
    model.eval()
    preds_all=[]
    labels_all=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            xb = xb.to(device)
            p = model(xb).cpu().numpy().flatten()
            preds_all.extend(p)
            labels_all.extend(yb.numpy().flatten())
    # binary threshold
    yhat = (np.array(preds_all) > 0.5).astype(int)
    print(f"Epoch {epoch} Acc {accuracy_score(labels_all,yhat):.4f} F1 {f1_score(labels_all,yhat, zero_division=0):.4f}")


<h2>Evaluation metrics utilities (including FPR, FNR, MCC)</h2>

In [ ]:
# Cell 11: compute detailed metrics
def compute_metrics(y_true, y_pred, y_score=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(set(y_true))>1 else np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp/(fp+tn) if (fp+tn) else np.nan
    fnr = fn/(fn+tp) if (fn+tp) else np.nan
    auc = roc_auc_score(y_true, y_score) if y_score is not None and len(np.unique(y_true))>1 else np.nan
    return {"accuracy":acc,"precision":prec,"recall":rec,"f1":f1,"mcc":mcc,"fpr":fpr,"fnr":fnr,"auc":auc}

# example
# y_score could be clf.predict_proba(X_test)[:,1]
print(compute_metrics(y_test, y_pred, y_score=None))


<h2>Jupyter mini UI: simple chat + recommendation demo</h2>

In [ ]:
# Cell 12: interactive demo using ipywidgets (run in Jupyter)
from ipywidgets import Text, Button, Output, VBox, Dropdown

query_input = Text(placeholder='Ask about a topic or request a summary/quiz...')
send_btn = Button(description="Ask")
out = Output()

def on_send(b):
    with out:
        out.clear_output()
        q = query_input.value
        retrieved = graph_rag_retrieve(q, top_k=5)
        prompt = build_rag_prompt(q, retrieved)
        print("Prompting LLM with grounded context (showing top sources):")
        for i,doc in enumerate(retrieved[:5]):
            print(f"[SRC_{i+1}] {doc.get('title','')}")
        # call OpenAI if configured:
        if OPENAI_API_KEY:
            ans = call_openai_chat(prompt)
            print("\n=== Answer ===\n", ans)
        else:
            print("\nOpenAI API key not configured. Show preview of prompt.\n")
            print(prompt[:1000])

send_btn.on_click(on_send)
VBox([query_input, send_btn, out])
